# CabinGuard-ADI — continuous cardiac monitor (v3), complete notebook

**What the model does.** Every 10 s it looks at the **last 60 s** of one ECG lead (steering wheel) and the PPG pulse (wheel sensor or camera) and answers: **"is there a cardiac emergency right now?"**
* **Emergency (label 1):** real **asystole, extreme bradycardia (< 40 bpm), ventricular tachycardia (VT) or ventricular fibrillation (VF)**.
* **No emergency (label 0):** normal heartbeat, false alarms (noise, loose electrode, movement), and real *tachycardia* (a fast but organised heartbeat, e.g. a stressed driver).

**Decision rule:** raise the emergency when **one window scores ≥ 0.5**, or **two windows in a row score ≥ the threshold** chosen on training data (section 7.2).

**Model:** an average of 5 LightGBM models (same settings, different random seeds) on 85 features. Averaging makes the result stable from run to run, and every model is trained in deterministic mode, so re-running the notebook gives the same numbers.

**Data (all open, PhysioNet):**

| dataset | what is used | role |
|---|---|---|
| CinC Challenge 2015 | 627 ICU alarms with ECG + PPG (5 alarm types), the minute before each alarm, and the earlier minutes of false-alarm records | events + normal signal |
| VTaC v1.0 | 4,673 ICU VT alarms with ECG + PPG (official patient split), and the earlier minutes of false alarms | VT events + normal signal + external test |
| Pulse Transit Time PPG | 22 healthy volunteers, ECG + finger PPG, sitting / walking / running | normal signal of healthy, moving people |

## Results on held-out test data (people, patients and alarms never used for training or for any choice)

| | v3 (this notebook) | v2 alarm checker, for reference |
|---|---|---|
| False triggers per hour, **healthy volunteers** sitting / walking / running (2.5 h) | **0** | 1.2 |
| False triggers per hour on ICU normal minutes, when both catch ≥ 95 % of real events | **10.0** | 15.1 |
| same, when both catch ≥ 90 % of real events | **5.9** | 10.7 |
| CinC alarms that are not emergencies (incl. real tachycardia), correctly ignored | **86 %** | 65 % |
| ROC-AUC, CinC alarms | **0.991** | 0.904 |
| ROC-AUC, VTaC VT alarms (unseen patients) | **0.941** | 0.932 |

**With the hybrid rule** (one window ≥ 0.5, or two in a row ≥ 0.17):
* **Real events detected:** 97.5 % of the VTaC test VTs, and all 8 real events in the 330-s CinC test records.
* **False triggers per hour:** 0 on healthy volunteers, about 6 on ICU patients whose alarm was false, and about 15 on VT-prone VTaC patients.

**Model:** 5 averaged LightGBM models on 85 features, about 90 ms per 10-s step on one CPU core.

## How to run (Google Colab)
1. Put **this notebook** and the **7 feature CSV files** (or the zip `cabinguard_v3_all_features.zip`) anywhere in your Google Drive, for example in the same folder.
2. Open the notebook with **Google Colaboratory**.
3. Click **Runtime → Run all**, then accept the Google Drive pop-up.
4. Wait about 5–10 minutes. The model `cabinguard_monitor_v3.pkl` is saved next to the feature files.

Without the zip, the notebook downloads VTaC and the healthy-volunteer recordings from PhysioNet (about 2 GB, 1–2 hours) and recomputes every feature (about 30 minutes).

# 0. Connect Google Drive and find the data
The first cell is the same in every notebook of the team: it mounts Google Drive (accept the pop-up) and finds the shared `AI` folder. One-time step for each person: in Google Drive, *Shared with me* → right-click `AI` → *Organize* → *Add shortcut* → *My Drive*.

The second cell looks in the shared `AI/datasets/` folder for:
* the **7 feature CSV files** (the folder that contains `features_vtac.csv`, normally `AI/datasets/data`). If it only finds `cabinguard_v3_all_features.zip`, it unzips it on Colab's disk;
* your **CinC 2015 folder** (the folder that contains the `ALARMS` file). It is used only if it holds every record; otherwise the few CinC files needed are downloaded from PhysioNet to Colab's disk.

The shared folder is read-only, so the results and the model are saved in **your own Drive**, in `MyDrive/AI_outputs/cardiac_arrest` (`OUT`).

In [2]:
# ===== 0. CHEMINS DU PROJET : exécuter cette cellule en premier =====
# Données : lues dans le dossier partagé « AI » (AI/datasets/), en lecture seule.
# Résultats : écrits dans VOTRE Drive, dans MyDrive/AI_outputs/<NOTEBOOK>.
#
# Une seule fois par personne, dans Google Drive (navigateur) :
#   « Partagés avec moi » > clic droit sur « AI » > Organiser > Ajouter un raccourci > « Mon Drive »
import glob, os
from pathlib import Path

NOTEBOOK = "cardiac_arrest"          # sous-dossier de sortie de ce notebook
BESOINS = ['cardiac_csv']              # datasets utilisés par ce notebook (clés de DATA ci-dessous)
AI_ROOT_MANUEL = None              # sur un PC uniquement, ex. r"G:\Mon Drive\AI" ; sinon laisser None

try:                               # Colab : monter Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    EN_COLAB = True
except ImportError:                # Jupyter sur un PC
    EN_COLAB = False


def trouver_racine_ai(nom="AI"):
    """Le dossier « AI » qui contient datasets/ et XAI/ (raccourci ou dossier dans Mon Drive, ou Drive partagé)."""
    ok = lambda p: (Path(p) / "datasets").is_dir() and (Path(p) / "XAI").is_dir()
    if AI_ROOT_MANUEL:
        return Path(AI_ROOT_MANUEL)
    for base in ("MyDrive", "Shareddrives"):
        for niveaux in range(4):                       # AI à la racine, puis jusqu'à 3 dossiers plus bas
            for p in sorted(glob.glob(f"/content/drive/{base}/{'*/' * niveaux}{nom}")):
                if ok(p):
                    return Path(p)
    for d in (Path.cwd(), *Path.cwd().parents):        # PC : notebook ouvert depuis le dossier AI synchronisé
        for p in (d, d / nom):
            if ok(p):
                return p
    raise FileNotFoundError(
        f"Dossier « {nom} » introuvable. Dans Google Drive : « Partagés avec moi » > clic droit sur « {nom} » "
        "> Organiser > Ajouter un raccourci > « Mon Drive », puis relancer cette cellule. "
        "Sur un PC : renseigner AI_ROOT_MANUEL.")


def _dossiers(racine, profondeur):
    """`racine` puis ses sous-dossiers, du moins profond au plus profond (dossiers cachés ignorés)."""
    niveau = [Path(racine)] if Path(racine).is_dir() else []
    for n in range(profondeur + 1):
        yield from niveau
        if n == profondeur:
            return
        suivant = []
        for d in niveau:
            try:
                with os.scandir(d) as entrees:
                    suivant += sorted(Path(e.path) for e in entrees
                                      if e.is_dir() and not e.name.startswith((".", "__")))
            except OSError:
                pass
        niveau = suivant


def trouver_fichier(noms, *dossiers, profondeur=3):
    """Premier fichier appelé `noms` (un nom ou une liste) dans `dossiers` ou leurs sous-dossiers."""
    noms = [noms] if isinstance(noms, str) else list(noms)
    for racine in dossiers:
        for d in _dossiers(racine, profondeur):
            for nom in noms:
                if (d / nom).is_file():
                    return d / nom
    raise FileNotFoundError(f"{' / '.join(noms)} introuvable dans : " + " ; ".join(map(str, dossiers)))


def trouver_dossier_contenant(sous_dossier, *dossiers, profondeur=3):
    """Premier dossier (dans `dossiers` ou leurs sous-dossiers) qui contient `sous_dossier`."""
    for racine in dossiers:
        for d in _dossiers(racine, profondeur):
            if (d / sous_dossier).is_dir():
                return d
    raise FileNotFoundError(f"Aucun dossier contenant {sous_dossier}/ dans : " + " ; ".join(map(str, dossiers)))


AI_ROOT = trouver_racine_ai()
DATASETS = AI_ROOT / "datasets"


def dataset(*mots, exact=False):
    """Sous-dossier de datasets/ dont le nom contient l'un des mots (ou vaut ce mot si exact=True), sans tenir compte des majuscules."""
    dossiers = sorted(d for d in DATASETS.iterdir() if d.is_dir())
    for mot in mots:
        for d in dossiers:
            if (d.name.lower() == mot.lower()) if exact else (mot.lower() in d.name.lower()):
                return d
    return DATASETS / mots[0]      # introuvable : signalé « MANQUANT » ci-dessous


DATA = {
    "seizure":     dataset("seizure"),                               # seizure dataset (release from github)
    "stroke":      dataset("neuroface", "stroke"),                   # NeuroFace Stroke dataset
    "cardiac":     dataset("heart"),                                 # heart diseasing dataset
    "cardiac_csv": dataset("cardiac_arrest_csv", "data", exact=True),  # CSV du modèle arrêt cardiaque
}

# Le dossier partagé est en lecture seule : chacun écrit ses résultats dans son propre Drive
SORTIES = (Path("/content/drive/MyDrive") if EN_COLAB else Path.cwd()) / "AI_outputs" / NOTEBOOK
SORTIES.mkdir(parents=True, exist_ok=True)

print("Dossier AI :", AI_ROOT)
for cle in BESOINS:
    print(f"  {'OK      ' if DATA[cle].is_dir() else 'MANQUANT'} DATA[{cle!r}] = {DATA[cle]}")
print("Sorties    :", SORTIES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Dossier « AI » introuvable. Dans Google Drive : « Partagés avec moi » > clic droit sur « AI » > Organiser > Ajouter un raccourci > « Mon Drive », puis relancer cette cellule. Sur un PC : renseigner AI_ROOT_MANUEL.

In [ ]:
import os, zipfile

# Paths of this notebook. The shared AI folder is read-only: everything the notebook writes goes to OUT.
LOCAL = "/content/data" if EN_COLAB else "./data"         # Colab's own disk, for anything downloaded
VTAC_DIR, PTT_DIR = f"{LOCAL}/vtac", f"{LOCAL}/ptt"
OUT = str(SORTIES)                                         # results + model: MyDrive/AI_outputs/cardiac_arrest (your Drive)
os.makedirs(OUT, exist_ok=True)

def find_dir_with(name, *roots):
    """Folder (one of roots or a sub-folder) that contains the file `name`, or None."""
    try:
        return str(trouver_fichier(name, *roots).parent)
    except FileNotFoundError:
        return None

# the 7 feature CSV files (read-only), normally in AI/datasets/data
FEAT_DIR = find_dir_with("features_vtac.csv", DATA["cardiac_csv"], DATASETS, AI_ROOT / "XAI")
if FEAT_DIR is None:                                       # only the zip is shared: unzip it on Colab's disk
    zip_dir = find_dir_with("cabinguard_v3_all_features.zip", DATA["cardiac_csv"], DATASETS, AI_ROOT / "XAI")
    if zip_dir:
        FEAT_DIR = f"{LOCAL}/features"
        zipfile.ZipFile(f"{zip_dir}/cabinguard_v3_all_features.zip").extractall(FEAT_DIR)
        print("features unzipped into", FEAT_DIR)

def feature_csv(name):
    """A feature file: the shared copy if it exists, otherwise your own copy in OUT (computed once)."""
    if FEAT_DIR and os.path.exists(f"{FEAT_DIR}/{name}.csv"):
        return f"{FEAT_DIR}/{name}.csv"
    return f"{OUT}/{name}.csv"

# CinC 2015 folder (the one with ALARMS). Used only if it holds every record, because nothing can be
# downloaded into the shared folder; otherwise the files needed are downloaded to Colab's disk.
CINC_DIR = find_dir_with("ALARMS", DATA["cardiac"], DATASETS)
if CINC_DIR:
    names = set(os.listdir(CINC_DIR))
    recs = open(f"{CINC_DIR}/RECORDS").read().split() if "RECORDS" in names else []
    if not recs or not all(f"{r}.hea" in names and f"{r}.mat" in names for r in recs):
        CINC_DIR = None
CINC_DIR = CINC_DIR or f"{LOCAL}/training"

print("CinC folder:", repr(CINC_DIR), "| found:", os.path.exists(f"{CINC_DIR}/ALARMS"))
print("feature folder:", repr(FEAT_DIR), "| results and model ->", repr(OUT))
n_csv = len([f for f in os.listdir(FEAT_DIR) if f.endswith(".csv")]) if FEAT_DIR else 0
print("feature files found:", n_csv, "of 7")
if n_csv < 7:
    print("WARNING: the feature CSV files were not found, so the notebook will download and recompute everything (1-2 h).\n"
          "Put the 7 files in AI/datasets/data, or set FEAT_DIR by hand here.")

# 1. Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ["wfdb", "lightgbm", "shap"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, time, json, pickle, warnings, urllib.request, platform
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import wfdb, lightgbm as lgb, sklearn
from joblib import Parallel, delayed
from scipy.signal import resample_poly
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

# paths (FEAT_DIR, CINC_DIR, VTAC_DIR, PTT_DIR, OUT) are set in section 0
# optional, only for the comparison in section 9: searched in the shared AI folder, then in your old MyDrive/cabinguard/outputs
V2_DIR = find_dir_with("cabinguard_cardiac_v2.pkl", DATA["cardiac_csv"], AI_ROOT / "XAI", "/content/drive/MyDrive/cabinguard/outputs")
V2_MODEL = f"{V2_DIR or OUT}/cabinguard_cardiac_v2.pkl"

N_JOBS = min(os.cpu_count() or 2, 4)
SEED, STRIDE = 42, 10                 # the monitor looks at the last 60 s every 10 s
EVENTS = ["Asystole", "Bradycardia", "Ventricular_Tachycardia", "Ventricular_Flutter_Fib"]
VTAC_WEIGHT = 0.25                    # VTaC alarms count less (7x more alarms than CinC, all of one type)
# LightGBM settings found by the Optuna search of the v2 notebook (36 trials, training data only)
PARAMS = dict(n_estimators=500, learning_rate=0.04551, num_leaves=59, min_child_samples=48, subsample=0.7123, subsample_freq=1,
              colsample_bytree=0.24141, reg_lambda=0.00344, reg_alpha=3.22119, class_weight="balanced",
              deterministic=True, force_col_wise=True)          # same result on every run
N_SEEDS = 5                           # the final model averages 5 LightGBM models trained with different random seeds

FEATURE_FILES = ["features_cinc", "features_vtac", "normal_cinc", "normal_vtac", "normal_ptt", "around_cinc", "around_vtac"]
FAST_MODE = all(os.path.exists(feature_csv(f)) for f in FEATURE_FILES)
print("fast mode (all features cached):", FAST_MODE, "| cores:", N_JOBS, "| lightgbm", lgb.__version__, "| wfdb", wfdb.__version__)

# 2. Feature code (one ECG lead + PPG, 60 s window)

| Group | # | What it measures |
|---|---|---|
| ECG rhythm | 25 | heart rate, beat count, irregularity, longest pause, time since last beat, fastest / slowest 4-beat run (baseline, last 16 / 10 / 5 s) |
| second beat detector | 3 | rate, pause and fastest run from an independent Pan-Tompkins-style detector |
| ECG quality / artefact | 13 | kurtosis, skewness, QRS-band power, slow drift, high-frequency noise, detector agreement, flat line, clipping, missing data |
| VF detectors | 8 | dominant frequency, spectral concentration, VF-band power, VF filter leakage, threshold crossings, Lempel-Ziv complexity |
| QRS shape | 6 | similarity of the last beats to the patient's baseline beat and to each other, QRS width |
| PPG pulse | 28 | pulse rate, pauses, irregularity, pulse size vs baseline, flat pulse, clipping, pulse shape |
| ECG ↔ PPG cross-check | 8 | share of heartbeats followed by a pulse, ECG / pulse rate ratio, beat-pulse count difference |

In [ ]:
"""CabinGuard cardiac features: ECG (one lead) + PPG only, from the 60 s before the alarm.

Everything here uses only signals a car can measure (one ECG lead from the steering
wheel, PPG from the wheel or a camera). Arterial pressure, respiration and the
monitor's alarm type are never used as inputs.
"""
import warnings
import numpy as np
import wfdb
from wfdb import processing
from scipy.signal import butter, sosfiltfilt, find_peaks, welch
from scipy.stats import kurtosis, skew

FS = 250
T0, T1 = 240, 300                 # analysis window in the record (s): 60 s ending at the alarm
BASE = (0.0, 44.0)                # baseline, 240-284 s (seconds relative to T0)
L16, L10, L5 = (44.0, 60.0), (50.0, 60.0), (55.0, 60.0)
ECG_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V", "MCL"]

In [ ]:
# ----------------------------------------------------------------------------- helpers
def _sos(lo, hi, order=3):
    return butter(order, [lo, hi], btype="band", fs=FS, output="sos")

SOS = {k: _sos(*k) for k in [(0.5, 40), (5, 15), (0.5, 8), (1, 30)]}

def bp(x, lo, hi):
    return sosfiltfilt(SOS[(lo, hi)], x)

def fill_nan(x):
    """Linear interpolation of missing samples. Returns (signal or None, missing fraction)."""
    x = np.asarray(x, float)
    m = np.isnan(x)
    if m.all():
        return None, 1.0
    if m.any():
        i = np.arange(len(x))
        x = x.copy()
        x[m] = np.interp(i[m], i[~m], x[~m])
    return x, float(m.mean())

def seg(x, win):
    return x[int(win[0] * FS):int(win[1] * FS)]

def band_power(f, p, lo, hi):
    m = (f >= lo) & (f < hi)
    return float(p[m].sum())

def block_stat(x, sec, fn):
    """Statistic per block of `sec` seconds, interpolated back to every sample."""
    n = int(sec * FS)
    k = max(len(x) // n, 1)
    vals = np.array([fn(x[i * n:(i + 1) * n]) for i in range(k)])
    centers = (np.arange(k) + 0.5) * n
    return np.interp(np.arange(len(x)), centers, vals), vals

def longest_run(mask):
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


# ----------------------------------------------------------------------------- lead handling
def lead_I(sig, names):
    """Lead I (left hand - right hand, the steering-wheel lead) when it exists or can be derived."""
    g = {n: sig[:, names.index(n)] for n in names}
    if "I" in g:
        return g["I"], "I"
    if "II" in g and "III" in g:
        return g["II"] - g["III"], "II-III"
    if "II" in g and "aVF" in g:
        return 2 * g["II"] - 2 * g["aVF"], "2II-2aVF"
    if "II" in g and "aVL" in g:
        return g["aVL"] + g["II"] / 2, "aVL+II/2"
    if "II" in g and "aVR" in g:
        return -2 * g["aVR"] - g["II"], "-2aVR-II"
    return None, None


def load_window(path, rec):
    r = wfdb.rdrecord(f"{path}/{rec}", sampfrom=T0 * FS, sampto=T1 * FS)
    return r.p_signal.astype(float), list(r.sig_name)

In [ ]:
# ----------------------------------------------------------------------------- beat detection
def qrs_xqrs(x):
    if x is None or np.ptp(x) < 1e-6:
        return np.array([], int)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return np.asarray(processing.xqrs_detect(sig=x, fs=FS, verbose=False), int)
    except Exception:
        return np.array([], int)


def qrs_energy(x):
    """Pan-Tompkins-style detector (independent of xqrs, used for detector agreement)."""
    if x is None or np.ptp(x) < 1e-6:
        return np.array([], int)
    f = bp(x, 5, 15)
    e = np.gradient(f) ** 2
    w = int(0.12 * FS)
    mwi = np.convolve(e, np.ones(w) / w, "same")
    th, blocks = block_stat(mwi, 2, lambda s: np.percentile(s, 98))
    floor = 0.1 * np.median(blocks)
    pk, _ = find_peaks(mwi, height=np.maximum(0.3 * th, floor), distance=int(0.18 * FS))
    # move each peak to the largest |ECG| sample nearby (aligns with the R wave)
    r = int(0.08 * FS)
    af = np.abs(f)
    return np.array([max(p - r, 0) + int(np.argmax(af[max(p - r, 0):p + r])) for p in pk], int)


def ppg_peaks(x):
    """Systolic peaks of the PPG; returns (peak indices, pulse amplitudes, filtered PPG)."""
    if x is None:
        return np.array([], int), np.array([]), None
    if np.ptp(x) < 1e-9:                                    # sensor flat for the whole minute
        return np.array([], int), np.array([]), np.zeros(len(x))
    f = bp(x, 0.5, 8)
    loc, _ = block_stat(f, 2, lambda s: np.percentile(s, 95) - np.percentile(s, 5))
    base = np.percentile(seg(f, BASE), 95) - np.percentile(seg(f, BASE), 5)
    prom = np.maximum(0.3 * loc, 0.1 * base)
    pk, _ = find_peaks(f, distance=int(0.27 * FS), prominence=prom)
    amps = []
    prev = 0
    for p in pk:
        lo = max(prev, p - int(0.8 * FS))
        amps.append(f[p] - f[lo:p + 1].min())
        prev = p
    return pk, np.asarray(amps, float), f


def match_f1(a, b, tol=0.15):
    """Agreement between two beat lists (seconds): F1 with a tolerance."""
    if len(a) == 0 and len(b) == 0:
        return np.nan
    if len(a) == 0 or len(b) == 0:
        return 0.0
    hit = sum(np.min(np.abs(b - t)) <= tol for t in a)
    return 2 * hit / (len(a) + len(b))

In [ ]:
# ----------------------------------------------------------------------------- feature groups
def rhythm(t, win, pre):
    a, b = win
    tt = t[(t >= a) & (t < b)]
    rr = np.diff(tt)
    o = {f"{pre}_n": len(tt) / (b - a) * 60}            # beats per minute counted
    o[f"{pre}_hr"] = 60 / np.median(rr) if len(rr) >= 2 else np.nan
    o[f"{pre}_cv"] = rr.std() / rr.mean() if len(rr) >= 3 else np.nan
    o[f"{pre}_maxgap"] = float(np.diff(np.r_[a, tt, b]).max())
    o[f"{pre}_endgap"] = float(b - tt[-1]) if len(tt) else float(b - a)
    if len(rr) >= 4:
        r4 = np.convolve(rr, np.ones(4) / 4, "valid")
        o[f"{pre}_hrmax4"], o[f"{pre}_hrmin4"] = 60 / r4.min(), 60 / r4.max()
    else:
        o[f"{pre}_hrmax4"] = o[f"{pre}_hrmin4"] = np.nan
    return o


def ecg_quality(x, win, pre):
    s = seg(x, win)
    f = bp(x, 0.5, 40)
    fs_ = seg(f, win)
    fr, p = welch(s - s.mean(), FS, nperseg=min(len(s), 4 * FS))
    tot = band_power(fr, p, 0.5, 100) + 1e-12
    o = {f"{pre}_kurt": float(kurtosis(fs_)), f"{pre}_skew": float(skew(fs_)),
         f"{pre}_psqi": band_power(fr, p, 5, 15) / (band_power(fr, p, 5, 40) + 1e-12),
         f"{pre}_bas": band_power(fr, p, 0, 1) / (band_power(fr, p, 0, 40) + 1e-12),
         f"{pre}_hf": band_power(fr, p, 40, 100) / tot}
    return o


def flat_seconds(f, win, ref):
    """Longest run (s) of 0.5-s blocks whose peak-to-peak is below 5% of the baseline size."""
    s = seg(f, win)
    n = int(0.5 * FS)
    k = len(s) // n
    if k == 0:
        return np.nan
    if ref <= 1e-9:                                         # flat already at baseline
        return (win[1] - win[0]) if np.ptp(s) <= 1e-9 else np.nan
    ptp = np.ptp(s[:k * n].reshape(k, n), axis=1)
    return longest_run(ptp < 0.05 * ref) * 0.5


def vf_features(x, win, pre):
    """Classic ventricular-fibrillation detectors on the last seconds of ECG."""
    s = seg(bp(x, 1, 30), win)
    s = s - s.mean()
    o = {}
    fr, p = welch(s, FS, nperseg=min(len(s), 4 * FS))
    m = (fr >= 0.5) & (fr <= 20)
    tot = p[m].sum() + 1e-12
    fd = fr[m][np.argmax(p[m])]
    o[f"{pre}_fdom"] = float(fd)
    o[f"{pre}_conc"] = float(p[(fr >= fd - 1) & (fr <= fd + 1)].sum() / tot)      # spectral concentration
    o[f"{pre}_vfband"] = float(p[(fr >= 2) & (fr < 8)].sum() / tot)
    # VF filter leakage (Kuo & Dillman): low leakage = sine-like = VF
    d = np.abs(np.diff(s)).sum() + 1e-12
    N = int(round(np.pi * np.abs(s).sum() / d / 2))
    if 1 <= N < len(s) // 2:
        o[f"{pre}_leak"] = float(np.abs(s[N:] + s[:-N]).sum() / (np.abs(s[N:]).sum() + np.abs(s[:-N]).sum() + 1e-12))
    else:
        o[f"{pre}_leak"] = np.nan
    # threshold-crossing sample count (Arafat): share of samples above 20% of max, per 3-s block
    n3 = 3 * FS
    blocks = [s[i:i + n3] for i in range(0, len(s) - n3 + 1, n3)] or [s]
    o[f"{pre}_tcsc"] = float(np.mean([np.mean(np.abs(b / (np.abs(b).max() + 1e-12)) > 0.2) for b in blocks]))
    # mean absolute value of the normalised signal (Anas) and zero crossings per second
    o[f"{pre}_mav"] = float(np.mean([np.mean(np.abs(b / (np.abs(b).max() + 1e-12))) for b in blocks]))
    o[f"{pre}_zc"] = float(np.sum(np.diff(np.sign(s)) != 0) / (len(s) / FS))
    # Lempel-Ziv complexity of the signal binarised at its median
    o[f"{pre}_lz"] = lz_complexity((s > np.median(s)).astype(np.int8))
    return o


def lz_complexity(b):
    """Normalised Lempel-Ziv complexity (Kaspar-Schuster)."""
    n = len(b)
    if n < 2:
        return np.nan
    s = b.tobytes()
    c, l, i, k, kmax = 1, 1, 0, 1, 1
    while True:
        if s[i + k - 1] == s[l + k - 1]:
            k += 1
            if l + k > n:
                c += 1
                break
        else:
            kmax = max(k, kmax)
            i += 1
            if i == l:
                c += 1
                l += kmax
                if l + 1 > n:
                    break
                i, k, kmax = 0, 1, 1
            else:
                k = 1
    return float(c * np.log2(n) / n)


def beat_morphology(f, beats, pre):
    """QRS shape: do the beats before the alarm look like the patient's baseline beats? (VT = wide, different)."""
    h = int(0.1 * FS)
    def cut(idx):
        return np.array([f[i - h:i + h] for i in idx if h <= i < len(f) - h])
    t = beats / FS
    bb, lb = beats[(t >= BASE[0]) & (t < BASE[1])], beats[(t >= L16[0]) & (t < L16[1])]
    B, L = cut(bb), cut(lb)
    o = {f"{pre}_corr": np.nan, f"{pre}_lowcorr": np.nan, f"{pre}_width": np.nan,
         f"{pre}_width_ratio": np.nan, f"{pre}_selfcorr": np.nan, f"{pre}_amp_ratio": np.nan}
    if len(B) < 5 or len(L) < 3:
        return o
    tmpl = np.median(B, axis=0)
    def corr(a, b):
        a, b = a - a.mean(), b - b.mean()
        return float(a @ b / (np.sqrt((a @ a) * (b @ b)) + 1e-12))
    cs = np.array([corr(b, tmpl) for b in L])
    o[f"{pre}_corr"] = float(np.median(cs))
    o[f"{pre}_lowcorr"] = float(np.mean(cs < 0.7))
    def width(w):
        e = np.abs(w - np.median(w))
        return float(np.sum(e > 0.5 * e.max()) / FS * 1000)        # ms above half of the peak
    lt = np.median(L, axis=0)
    o[f"{pre}_width"] = width(lt)
    o[f"{pre}_width_ratio"] = width(lt) / (width(tmpl) + 1e-9)
    o[f"{pre}_selfcorr"] = float(np.median([corr(L[i], L[i + 1]) for i in range(len(L) - 1)])) if len(L) > 1 else np.nan
    o[f"{pre}_amp_ratio"] = float(np.ptp(lt) / (np.ptp(tmpl) + 1e-9))
    return o


def ppg_morphology(f, pk, pre):
    """Do the pulses before the alarm keep the baseline pulse shape? (lost pulse / motion = no)."""
    o = {f"{pre}_tcorr": np.nan, f"{pre}_skew10": np.nan}
    if f is None:
        return o
    o[f"{pre}_skew10"] = float(skew(seg(f, L10)))
    h0, h1 = int(0.3 * FS), int(0.5 * FS)
    t = pk / FS
    def cut(idx):
        return np.array([f[i - h0:i + h1] for i in idx if h0 <= i < len(f) - h1])
    B = cut(pk[(t >= BASE[0]) & (t < BASE[1])])
    L = cut(pk[(t >= L10[0]) & (t < L10[1])])
    if len(B) >= 5 and len(L) >= 2:
        tm = np.median(B, axis=0)
        cs = [np.corrcoef(b, tm)[0, 1] for b in L]
        o[f"{pre}_tcorr"] = float(np.nanmedian(cs))
    return o

In [ ]:
# ----------------------------------------------------------------------------- one record
def ecg_block(ecg_raw):
    """All features that need only the ECG lead."""
    ecg, ecg_nan = fill_nan(ecg_raw)
    o = {"ecg_nan": ecg_nan}
    if ecg is None:
        return o, None
    bx = qrs_xqrs(ecg)
    be = qrs_energy(ecg)
    tx, te = bx / FS, be / FS
    f = bp(ecg, 0.5, 40)
    for win, tag in [(BASE, "b"), (L16, "16"), (L10, "10"), (L5, "5")]:
        r = rhythm(tx, win, f"ecg{tag}")
        if tag in ("b", "5"):
            r = {k: v for k, v in r.items() if k.endswith(("_hr", "_n"))}
        o.update(r)
    # beats from the second detector in the last 16 s (robust rate when one detector fails)
    o.update({k.replace("ecg16", "ecgE16"): v for k, v in rhythm(te, L16, "ecg16").items()
              if k.endswith(("_hr", "_maxgap", "_hrmax4"))})
    o["ecg_bsqi16"] = match_f1(tx[tx >= L16[0]], te[te >= L16[0]])
    o["ecg_bsqi_b"] = match_f1(tx[tx < BASE[1]], te[te < BASE[1]])
    o.update(ecg_quality(ecg, L10, "ecgq10"))
    o.update({k: v for k, v in ecg_quality(ecg, BASE, "ecgqb").items() if k.endswith(("kurt", "psqi"))})
    ref = np.percentile(seg(f, BASE), 99) - np.percentile(seg(f, BASE), 1)
    o["ecg_amp_ratio10"] = (np.percentile(seg(f, L10), 99) - np.percentile(seg(f, L10), 1)) / (ref + 1e-9)
    o["ecg_flat16"] = flat_seconds(f, L16, ref)
    raw16 = seg(ecg, L16)
    o["ecg_sat16"] = float(np.mean((raw16 == raw16.max()) | (raw16 == raw16.min())))
    o.update(vf_features(ecg, (52.0, 60.0), "vf8"))
    o.update(beat_morphology(f, bx, "qrs"))
    o["ecg_hr_jump"] = o["ecg10_hr"] - o["ecgb_hr"]
    return o, tx


def ppg_block(ppg_raw):
    ppg, ppg_nan = fill_nan(ppg_raw)
    o = {"ppg_nan": ppg_nan}
    if ppg is None:
        return o, None
    pk, amps, f = ppg_peaks(ppg)
    tp = pk / FS
    for win, tag in [(BASE, "b"), (L16, "16"), (L10, "10"), (L5, "5")]:
        r = rhythm(tp, win, f"ppg{tag}")
        if tag in ("b", "5"):
            r = {k: v for k, v in r.items() if k.endswith(("_hr", "_n"))}
        o.update(r)
    def med_amp(win):
        m = (tp >= win[0]) & (tp < win[1])
        return float(np.median(amps[m])) if m.sum() else 0.0
    ab = med_amp(BASE)
    o["ppg_pamp10"] = med_amp(L10) / (ab + 1e-9) if ab > 0 else np.nan     # pulse size now vs baseline
    o["ppg_pamp5"] = med_amp(L5) / (ab + 1e-9) if ab > 0 else np.nan
    ref = np.percentile(seg(f, BASE), 95) - np.percentile(seg(f, BASE), 5)
    o["ppg_std_ratio10"] = float(seg(f, L10).std() / (seg(f, BASE).std() + 1e-9))
    o["ppg_flat16"] = flat_seconds(f, L16, ref)
    raw16 = seg(ppg, L16)
    o["ppg_sat16"] = float(np.mean((raw16 == raw16.max()) | (raw16 == raw16.min())))
    o["ppg_kurt10"] = float(kurtosis(seg(f, L10)))
    o.update(ppg_morphology(f, pk, "ppgm"))
    o["ppg_hr_jump"] = o["ppg10_hr"] - o["ppgb_hr"]
    return o, tp


def cross_block(te, tp):
    """Does every heartbeat produce a pulse? (the core ECG + PPG cross-check)"""
    o = {k: np.nan for k in ["x_match16", "x_match_b", "x_match_drop", "x_ptt", "x_ptt_iqr",
                             "x_rate_ratio10", "x_rate_diff10", "x_count_diff16"]}
    if te is None or tp is None:
        return o
    def next_delay(e):
        if len(tp) == 0:
            return np.full(len(e), np.inf)
        j = np.searchsorted(tp, e)
        ok = j < len(tp)
        d = np.full(len(e), np.inf)
        d[ok] = tp[j[ok]] - e[ok]
        return d
    # the beat-to-pulse delay differs between patients and monitors: learn it from the baseline
    eb = te[(te >= BASE[0]) & (te < BASE[1])]
    db = next_delay(eb)
    db = db[db < 1.5]
    D = float(np.median(db)) if len(db) >= 5 else 0.3
    def match(win, tol=0.15):
        e = te[(te >= win[0]) & (te < win[1])]
        if len(e) == 0:
            return np.nan, np.array([])
        if len(tp) == 0:
            return 0.0, np.array([])
        d = np.min(np.abs(tp[None, :] - (e[:, None] + D)), axis=1)
        good = d <= tol
        return float(good.mean()), next_delay(e[good])
    o["x_match16"], d16 = match(L16)
    o["x_match_b"], _ = match(BASE)
    o["x_match_drop"] = o["x_match_b"] - o["x_match16"]
    d16 = d16[np.isfinite(d16)]
    if len(d16) >= 3:
        o["x_ptt"] = float(np.median(d16) - D)
        o["x_ptt_iqr"] = float(np.subtract(*np.percentile(d16, [75, 25])))
    re_ = rhythm(te, L10, "e")["e_hr"]
    rp_ = rhythm(tp, L10, "p")["p_hr"]
    if np.isfinite(re_) and np.isfinite(rp_):
        o["x_rate_ratio10"] = re_ / rp_
        o["x_rate_diff10"] = abs(re_ - rp_)
    o["x_count_diff16"] = (np.sum(te >= L16[0]) - np.sum(tp >= L16[0])) / 16 * 60
    return o


def record_features(sig, names, ecg_lead):
    """Features for one record using ONE ECG lead (`ecg_lead`, or 'I*' for derived lead I) + PLETH."""
    if ecg_lead == "I*":
        ecg_raw, _ = lead_I(sig, names)
    elif ecg_lead is None:
        ecg_raw = None
    else:
        ecg_raw = sig[:, names.index(ecg_lead)]
    ppg_raw = sig[:, names.index("PLETH")] if "PLETH" in names else None
    o = {}
    te = tp = None
    if ecg_raw is not None:
        e, te = ecg_block(ecg_raw)
        o.update(e)
    p, tp = ppg_block(ppg_raw)
    o.update(p)
    o.update(cross_block(te, tp))
    return o


# ----------------------------------------------------------------------------- VTaC (external VT dataset)
def load_vtac_window(hea_path, npy_path):
    """VTaC: the 240-300 s window saved as raw int16 + its header -> (physical signals, channel names)."""
    L = [l for l in open(hea_path).read().splitlines() if l and not l.startswith("#")]
    nsig = int(L[0].split()[1])
    raw = np.load(npy_path).astype(float)
    names, sig = [], np.empty_like(raw)
    for j, l in enumerate(L[1:1 + nsig]):
        p = l.split()
        g = p[2].split("/")[0]
        base = 0.0
        if "(" in g:
            g, b = g.split("(")
            base = float(b.rstrip(")"))
        gain = float(g) or 200.0
        col = raw[:, j]
        sig[:, j] = np.where(col == -32768, np.nan, (col - base) / gain)
        names.append(p[-1])
    return sig, names

# 3. Data
Small index files are always downloaded. The signals are downloaded only when the feature files are not cached (no fast mode). VTaC is read with HTTP byte ranges, so only the needed seconds are downloaded: 240–300 s (the alarm minute) for every alarm with PPG, 120–240 s for false alarms and test alarms, and 300–320 s for test alarms.

In [ ]:
CINC_URL = "https://physionet.org/files/challenge-2015/1.0.0/training"
VTAC_URL = "https://physionet.org/files/vtac/1.0"
PTT_URL = "https://physionet.org/files/pulse-transit-time-ppg/1.1.0"

def download(url, dest, resume_to=None):
    """Download with retries; resume_to = expected size (resumes a partial file)."""
    for k in range(10):
        have = os.path.getsize(dest) if os.path.exists(dest) else 0
        if have > 0 and (resume_to is None or have == resume_to):
            return True
        try:
            hdr = {"Range": f"bytes={have}-"} if (resume_to and have) else {}
            with urllib.request.urlopen(urllib.request.Request(url, headers=hdr), timeout=120) as r, open(dest, "ab" if hdr else "wb") as f:
                while chunk := r.read(1 << 16):
                    f.write(chunk)
        except Exception:
            time.sleep(2 + 3 * k)
    return os.path.exists(dest)

def vtac_header(ev):
    L = [l for l in open(f"{VTAC_DIR}/hea/{ev}.hea").read().splitlines() if l and not l.startswith("#")]
    return int(L[0].split()[1]), [l.split()[-1] for l in L[1:1 + int(L[0].split()[1])]]

def vtac_range(rec, sub, s0, s1):
    """Seconds s0-s1 of a VTaC record (format 16, 250 Hz) -> VTAC_DIR/sub/<event>.npy (only if it has PPG)."""
    ev = os.path.basename(rec); out = f"{VTAC_DIR}/{sub}/{ev}.npy"
    if os.path.exists(out):
        return
    if not os.path.exists(f"{VTAC_DIR}/hea/{ev}.hea"):
        download(f"{VTAC_URL}/{rec}.hea", f"{VTAC_DIR}/hea/{ev}.hea")
    nsig, names = vtac_header(ev)
    if "PLETH" not in names:
        return
    a, b = s0 * 250 * nsig * 2, s1 * 250 * nsig * 2 - 1
    for k in range(8):
        try:
            data = urllib.request.urlopen(urllib.request.Request(f"{VTAC_URL}/{rec}.dat", headers={"Range": f"bytes={a}-{b}"}), timeout=90).read()
            assert len(data) == b - a + 1
            np.save(out, np.frombuffer(data, dtype="<i2").reshape(-1, nsig)); return
        except Exception:
            time.sleep(2 + 3 * k)

for d in [CINC_DIR, PTT_DIR] + [f"{VTAC_DIR}/{s}" for s in ["hea", "win", "pre", "post"]]:
    os.makedirs(d, exist_ok=True)
for f, url in [(f"{CINC_DIR}/RECORDS", f"{CINC_URL}/RECORDS"), (f"{CINC_DIR}/ALARMS", f"{CINC_URL}/ALARMS"),
               (f"{VTAC_DIR}/RECORDS", f"{VTAC_URL}/RECORDS"), (f"{VTAC_DIR}/event_labels.csv", f"{VTAC_URL}/event_labels.csv"),
               (f"{VTAC_DIR}/benchmark_data_split.csv", f"{VTAC_URL}/benchmark_data_split.csv"), (f"{PTT_DIR}/RECORDS", f"{PTT_URL}/RECORDS")]:
    download(url, f)

cinc_records = open(f"{CINC_DIR}/RECORDS").read().split()
alarms = pd.read_csv(f"{CINC_DIR}/ALARMS", header=None, names=["record", "alarm", "label"])
vt_meta = pd.read_csv(f"{VTAC_DIR}/event_labels.csv").merge(pd.read_csv(f"{VTAC_DIR}/benchmark_data_split.csv"), on=["record", "event"])
vt_meta["label"] = (vt_meta.decision.astype(str).str.lower() == "true").astype(int)
vt_paths = {os.path.basename(r): r for r in open(f"{VTAC_DIR}/RECORDS").read().split()}
ptt_records = open(f"{PTT_DIR}/RECORDS").read().split()

if not FAST_MODE:
    t = time.time()
    todo = [f"{r}.{e}" for r in cinc_records for e in ("hea", "mat") if not os.path.exists(f"{CINC_DIR}/{r}.{e}")]
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(lambda f: download(f"{CINC_URL}/{f}", f"{CINC_DIR}/{f}"), todo))
    print(f"CinC ready ({time.time() - t:.0f} s)"); t = time.time()
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(lambda e: vtac_range(vt_paths[e], "win", 240, 300), vt_meta.event))
        list(ex.map(lambda e: vtac_range(vt_paths[e], "pre", 120, 240), vt_meta[(vt_meta.label == 0) | (vt_meta.split == "test")].event))
        list(ex.map(lambda e: vtac_range(vt_paths[e], "post", 300, 320), vt_meta[vt_meta.split == "test"].event))
    print(f"VTaC ready ({time.time() - t:.0f} s)"); t = time.time()
    def ptt_record(r):
        download(f"{PTT_URL}/{r}.hea", f"{PTT_DIR}/{r}.hea"); download(f"{PTT_URL}/{r}.atr", f"{PTT_DIR}/{r}.atr")
        h = wfdb.rdheader(f"{PTT_DIR}/{r}")
        download(f"{PTT_URL}/{r}.dat", f"{PTT_DIR}/{r}.dat", resume_to=h.sig_len * h.n_sig * 3 // 2)   # format 212
    with ThreadPoolExecutor(3) as ex:
        list(ex.map(ptt_record, ptt_records))
    print(f"healthy volunteers ready ({time.time() - t:.0f} s)")
print(len(cinc_records), "CinC alarms |", len(vt_meta), "VTaC alarms |", len(ptt_records), "healthy-volunteer recordings")

## 3.1 The healthy-volunteer PPG is upside down
The finger sensor of the Pulse Transit Time dataset measures reflected light, which *drops* when the pulse arrives; ICU monitors show the pulse upwards. In the raw signal below, the steep edge just after each heartbeat (red lines) goes **down**, so this PPG is inverted before features are computed.

In [ ]:
def load_ptt(rec, invert=True):
    r = wfdb.rdrecord(f"{PTT_DIR}/{rec}"); n = list(r.sig_name)
    ecg = resample_poly(r.p_signal[:, n.index("ecg")], 1, 2)                                 # 500 -> 250 Hz
    ppg = resample_poly(r.p_signal[:, n.index("pleth_2")], 1, 2) * (-1 if invert else 1)     # infrared, fingertip
    return np.c_[ecg, ppg], ["ECG", "PLETH"]

def ensure_ptt(rec):
    download(f"{PTT_URL}/{rec}.hea", f"{PTT_DIR}/{rec}.hea"); download(f"{PTT_URL}/{rec}.atr", f"{PTT_DIR}/{rec}.atr")
    h = wfdb.rdheader(f"{PTT_DIR}/{rec}")
    download(f"{PTT_URL}/{rec}.dat", f"{PTT_DIR}/{rec}.dat", resume_to=h.sig_len * h.n_sig * 3 // 2)

ensure_ptt("s1_sit")
sig, _ = load_ptt("s1_sit", invert=False)
beats_ = wfdb.rdann(f"{PTT_DIR}/s1_sit", "atr").sample // 2
t = np.arange(len(sig)) / FS; m = (t >= 60) & (t < 66)
fig, ax = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True)
ax[0].plot(t[m], bp(sig[:, 0], 0.5, 40)[m], c="k", lw=.7); ax[0].set_title("s1_sit: ECG")
ax[1].plot(t[m], bp(sig[:, 1], 0.5, 8)[m]); ax[1].set_title("raw PPG (before inversion)")
for b in beats_[(beats_ >= 60 * FS) & (beats_ < 66 * FS)] / FS:
    ax[1].axvline(b, c="r", lw=.6)
plt.tight_layout(); plt.show()

# 4. Feature extraction (read from `AI/datasets/data`, or computed once into `OUT`)
| file | windows | used for |
|---|---|---|
| `features_cinc` | the 60 s before each CinC alarm, once per ECG lead (+ lead I derived from limb leads, test only) | events / non-events |
| `features_vtac` | the 60 s before each VTaC alarm, leads I, II and V | VT events / false VT |
| `normal_cinc` | windows ending every 10 s from 60 to 240 s | normal signal (false-alarm records only) |
| `normal_vtac` | windows ending at 180 and 240 s (every 10 s for test alarms) | normal signal (false alarms only) |
| `normal_ptt` | every 10 s over each healthy-volunteer recording | normal signal of healthy, moving people |
| `around_cinc`, `around_vtac` | windows ending 280–330 s around each **test** alarm | checking whether a detection persists |

Training on **every available ECG lead** makes the model independent of the lead. The car will use lead I.

In [ ]:
def cinc_alarm_rows(rec):
    sig, names = load_window(CINC_DIR, rec)
    if "PLETH" not in names:
        return []
    ecgs = [n for n in names if n in ECG_LEADS and not np.isnan(sig[:, names.index(n)]).all()]
    primary = "II" if "II" in ecgs else (ecgs[0] if ecgs else None)
    rows = [{"record": rec, "lead": l, "primary": l == primary, **record_features(sig, names, l)} for l in ecgs]
    li, how = lead_I(sig, names)
    if li is not None and how != "I":
        rows.append({"record": rec, "lead": "I*", "primary": False, **record_features(sig, names, "I*")})
    if not ecgs:
        rows.append({"record": rec, "lead": "none", "primary": True, **record_features(sig, names, None)})
    return rows

def vtac_alarm_rows(ev):
    f = f"{VTAC_DIR}/win/{ev}.npy"
    if not os.path.exists(f):
        return []
    sig, names = load_vtac_window(f"{VTAC_DIR}/hea/{ev}.hea", f)
    ecgs = [n for n in ["I", "II", "V"] if n in names and not np.isnan(sig[:, names.index(n)]).all()]
    primary = "II" if "II" in ecgs else (ecgs[0] if ecgs else None)
    return [{"record": ev, "lead": l, "primary": l == primary, **record_features(sig, names, l)} for l in ecgs]

def cinc_normal(rec):
    r = wfdb.rdrecord(f"{CINC_DIR}/{rec}", sampto=240 * FS); sig, names = r.p_signal.astype(float), list(r.sig_name)
    if "PLETH" not in names:
        return []
    ecgs = [n for n in names if n in ECG_LEADS and not np.isnan(sig[:, names.index(n)]).all()]
    lead = "II" if "II" in ecgs else (ecgs[0] if ecgs else None)
    return [{"record": rec, "t_end": e, "lead": lead or "none", **record_features(sig[(e - 60) * FS:e * FS], names, lead)} for e in range(60, 241, STRIDE)]

def vtac_normal(ev, split):
    f = f"{VTAC_DIR}/pre/{ev}.npy"
    if not os.path.exists(f):
        return []
    sig, names = load_vtac_window(f"{VTAC_DIR}/hea/{ev}.hea", f)                                 # 120-240 s
    ecgs = [n for n in ["II", "I", "V"] if n in names and not np.isnan(sig[:, names.index(n)]).all()]
    if not ecgs:
        return []
    ends = range(180, 241, STRIDE) if split == "test" else (180, 240)
    return [{"record": ev, "t_end": e, "lead": ecgs[0], **record_features(sig[(e - 180) * FS:(e - 120) * FS], names, ecgs[0])} for e in ends]

def ptt_windows(rec):
    sig, names = load_ptt(rec)
    return [{"record": rec, "subject": rec.split("_")[0], "activity": rec.split("_")[1], "t_end": e, "lead": "ECG",
             **record_features(sig[(e - 60) * FS:e * FS], names, "ECG")} for e in range(60, len(sig) // FS + 1, STRIDE)]

def around_cinc(rec, lead):
    r = wfdb.rdrecord(f"{CINC_DIR}/{rec}"); sig, names = r.p_signal.astype(float), list(r.sig_name)
    return [{"record": rec, "t_end": e, "lead": lead, **record_features(sig[(e - 60) * FS:e * FS], names, None if lead == "none" else lead)}
            for e in range(280, min(len(sig) // FS, 330) + 1, STRIDE)]

def around_vtac(ev):
    parts = []
    for sub in ["pre", "win", "post"]:
        f = f"{VTAC_DIR}/{sub}/{ev}.npy"
        if not os.path.exists(f):
            return []
        parts.append(load_vtac_window(f"{VTAC_DIR}/hea/{ev}.hea", f))
    sig, names = np.vstack([p[0] for p in parts]), parts[0][1]                                    # 120-320 s
    ecgs = [n for n in ["II", "I", "V"] if n in names and not np.isnan(sig[:, names.index(n)]).all()]
    return [] if not ecgs else [{"record": ev, "t_end": e, "lead": ecgs[0], **record_features(sig[(e - 180) * FS:(e - 120) * FS], names, ecgs[0])}
                                for e in range(280, 321, STRIDE)]

def cached(name, fn, items):
    path = feature_csv(name)                                  # shared copy, else your own copy in OUT
    if os.path.exists(path):
        df = pd.read_csv(path); print(f"{name:15s} {len(df):6d} windows (cached)")
        return df
    t = time.time()
    out = Parallel(n_jobs=N_JOBS, batch_size=4)(delayed(fn)(*it) for it in items)
    df = pd.DataFrame([r for rows in out for r in rows]); df.to_csv(path, index=False)
    print(f"{name:15s} {len(df):6d} windows extracted in {time.time() - t:.0f} s")
    return df

# alarm windows (same as v2)
cinc_alarm = cached("features_cinc", cinc_alarm_rows, [(r,) for r in cinc_records])
if "alarm" not in cinc_alarm:
    cinc_alarm = cinc_alarm.merge(alarms[["record", "alarm", "label"]], on="record"); cinc_alarm.to_csv(f"{OUT}/features_cinc.csv", index=False)
has_win = [e for e in vt_meta.event if os.path.exists(f"{VTAC_DIR}/win/{e}.npy")] if not os.path.exists(feature_csv("features_vtac")) else []
vtac_alarm = cached("features_vtac", vtac_alarm_rows, [(e,) for e in has_win])
if "label" not in vtac_alarm:
    vtac_alarm = vtac_alarm.merge(vt_meta[["event", "record", "split", "label"]].rename(columns={"record": "patient", "event": "record"}), on="record")
    vtac_alarm.to_csv(f"{OUT}/features_vtac.csv", index=False)
FEATURES = [c for c in cinc_alarm.columns if c not in ("record", "lead", "primary", "alarm", "label")]
assert len(FEATURES) == 85, len(FEATURES)

# the frozen CinC test set (20 %, stratified by alarm type x label) -- the same as in v2
prim = cinc_alarm[cinc_alarm.primary].copy(); prim["strat"] = prim.alarm + "_" + prim.label.astype(str)
_, te_rec = train_test_split(prim.record, test_size=0.2, stratify=prim.strat, random_state=SEED)
CINC_TEST = set(te_rec)

# normal and around-the-alarm windows
n_cinc = cached("normal_cinc", cinc_normal, [(r,) for r in cinc_records])
n_vtac = cached("normal_vtac", vtac_normal, [(r.event, r.split) for r in vt_meta[(vt_meta.label == 0) | (vt_meta.split == "test")].itertuples()])
n_ptt = cached("normal_ptt", ptt_windows, [(r,) for r in ptt_records])
a_cinc = cached("around_cinc", around_cinc, [(r.record, r.lead) for r in prim[prim.record.isin(CINC_TEST)].itertuples()])
a_vtac = cached("around_vtac", around_vtac, [(e,) for e in vt_meta[vt_meta.split == "test"].event])
print(f"\n{len(FEATURES)} features | CinC alarms {prim.shape[0]} ({len(CINC_TEST)} test) | VTaC alarms {vtac_alarm.record.nunique()} | "
      f"healthy-volunteer signal {len(n_ptt) * STRIDE / 3600:.1f} h")

# 5. Labels, weights and splits
| source | label 1 | label 0 | weight | test part |
|---|---|---|---|---|
| CinC alarms (all ECG leads) | real asystole / bradycardia / VT / VF | false alarms **and real tachycardia** | 1 | frozen 20 % of records |
| VTaC alarms (leads I, II, V) | real VT | false VT | 0.25 | official patient-level test split |
| CinC normal minutes | — | all | 1 × `W_NORMAL` | same records as above |
| VTaC normal minutes | — | all | 0.25 × `W_NORMAL` | same patients as above |
| healthy volunteers | — | all | 1 × `W_NORMAL` | 7 of 22 volunteers held out |

**Only false-alarm records give "normal" minutes.** Before a *real* asystole or VF, most earlier windows already look abnormal (60–65 % flagged by the v2 model), because the patient is already deteriorating. Cross-validation folds are split by **record / patient / volunteer**, so no person is ever on both sides.

In [ ]:
PTT_TEST = ["s3", "s7", "s10", "s14", "s17", "s20", "s22"]

c = cinc_alarm[cinc_alarm.lead != "I*"].copy()
c["split"] = np.where(c.record.isin(CINC_TEST), "test", "train")
c["y"] = ((c.label == 1) & c.alarm.isin(EVENTS)).astype(int)
c = c.assign(kind="alarm", source="CinC", group="c_" + c.record, w=1.0)
v = vtac_alarm.copy()
v["split"] = np.where(v.split == "test", "test", "train")
v = v.assign(y=v.label, alarm="Ventricular_Tachycardia", kind="alarm", source="VTaC", group="v_" + v.patient.astype(str), w=VTAC_WEIGHT)
n_cinc = n_cinc.merge(prim[["record", "alarm", "label"]], on="record")
nc = n_cinc[n_cinc.label == 0].copy()
nc["split"] = np.where(nc.record.isin(CINC_TEST), "test", "train")
nc = nc.assign(y=0, kind="normal", source="CinC normal", group="c_" + nc.record, w=1.0, primary=True)
nv = n_vtac.merge(vt_meta[["event", "record", "split", "label"]].rename(columns={"record": "patient", "event": "record", "label": "alarm_label"}), on="record")
nv = nv[nv.alarm_label == 0].copy()
nv["split"] = np.where(nv.split == "test", "test", "train")
nv = nv.assign(y=0, kind="normal", source="VTaC normal", group="v_" + nv.patient.astype(str), w=VTAC_WEIGHT, primary=True, alarm="none")
npt = n_ptt.copy()
npt["split"] = np.where(npt.subject.isin(PTT_TEST), "test", "train")
npt = npt.assign(y=0, kind="normal", source="healthy volunteers", group="p_" + npt.subject, w=1.0, primary=True, alarm="none")
ALL = pd.concat([c, v, nc, nv, npt], ignore_index=True)
TR, TE = ALL[ALL.split == "train"].reset_index(drop=True), ALL[ALL.split == "test"].reset_index(drop=True)
summary = ALL.groupby(["source", "split"]).agg(rows=("y", "size"), events=("y", "sum")).unstack()
for sp, D_ in [("train", TR), ("test", TE)]:
    summary[("hours of normal signal", sp)] = [np.nan if s in ("CinC", "VTaC") else round((D_.source == s).sum() * STRIDE / 3600, 1) for s in summary.index]
summary

# 6. Helpers

In [ ]:
GRID = np.round(np.arange(0.01, 0.99, 0.01), 3)
HI = 0.5                                  # one window above this = emergency at once ("more likely than not")

class SeedEnsemble:
    """Average of N_SEEDS LightGBM models that differ only by their random seed (more stable than one model)."""
    def __init__(self, models):
        self.models = models
    def predict_proba(self, X):
        p = np.mean([m.predict_proba(X)[:, 1] for m in self.models], axis=0)
        return np.c_[1 - p, p]

def fit_model(X, y, w, n_seeds=N_SEEDS):
    return SeedEnsemble([lgb.LGBMClassifier(**PARAMS, random_state=SEED + k, n_jobs=N_JOBS, verbose=-1).fit(X, y, sample_weight=w)
                         for k in range(n_seeds)])

def weights(d, w_normal):
    w = d.w.to_numpy().copy(); w[(d.kind == "normal").to_numpy()] *= w_normal
    return w

def oof(d, w_normal, seed=0):
    """Out-of-fold predictions; folds never split a record / patient / volunteer."""
    p = np.zeros(len(d))
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(d, d.y, d.group):
        m = fit_model(d.iloc[a][FEATURES].to_numpy(float), d.y.iloc[a].to_numpy(), weights(d.iloc[a], w_normal))
        p[b] = m.predict_proba(d.iloc[b][FEATURES].to_numpy(float))[:, 1]
    return p

def ev_mask(d, src):
    return ((d.source == src) & (d.kind == "alarm") & d.primary).to_numpy()

def pick_threshold(d, p, target=0.95):
    """Highest threshold that still catches >= target of the real events in CinC AND in VTaC."""
    best = GRID[0]
    for t in GRID:
        if all((p[ev_mask(d, s) & (d.y == 1).to_numpy()] >= t).mean() >= target for s in ["CinC", "VTaC"]):
            best = t
    return float(best)

def rule_fires(seq, thr, rule, hi=HI):
    """seq = [(t_end, p), ...] in time order; yields (t_end, emergency?) for each window."""
    run = 0
    for t_end, pp in seq:
        run = run + 1 if pp >= thr else 0
        yield t_end, {"1 window": run >= 1, "2 in a row": run >= 2, "hybrid": pp >= hi or run >= 2}[rule]

def rule_per_hour(d, p, thr, rule="1 window", hi=HI):
    """False triggers per hour: one trigger = one new emergency episode in a recording."""
    d = d.assign(p=p).sort_values(["record", "t_end"]); n = 0
    for _, g in d.groupby("record"):
        fired = False
        for (_, f), pp in zip(rule_fires(zip(g.t_end, g.p), thr, rule, hi), g.p):
            if f and not fired:
                n += 1; fired = True
            if pp < thr:
                fired = False
    return n / (len(d) * STRIDE / 3600)

def scorecard(d, p, thr, name, dense=("CinC normal", "VTaC normal", "healthy volunteers")):
    """dense = normal sources sampled every 10 s. VTaC training alarms only have 2 normal windows 60 s apart,
    so for them only the share of flagged windows is meaningful."""
    row = {"model": name, "threshold": thr}
    for s in ["CinC", "VTaC"]:
        m = ev_mask(d, s); y = d.y.to_numpy()[m]
        row[f"{s} AUC"] = roc_auc_score(y, p[m])
        row[f"{s} events caught"] = (p[m][y == 1] >= thr).mean()
        row[f"{s} non-emergency alarms ignored"] = (p[m][y == 0] < thr).mean()
    for s in ["CinC normal", "VTaC normal", "healthy volunteers"]:
        m = (d.source == s).to_numpy()
        row[f"{s}: share of windows flagged"] = (p[m] >= thr).mean()
        if s in dense:
            row[f"{s}: false triggers/h"] = rule_per_hour(d[m], p[m], thr)
    return row

# 7. Training
## 7.1 How much should normal windows count?
Each value is scored on out-of-fold predictions for the training part, at the threshold that keeps ≥ 95 % of real events in both CinC and VTaC. The value with the smallest share of normal windows flagged (averaged over the three normal sources) is kept.

In [ ]:
cv, OOF = [], {}
t = time.time()
for wn in [0.03, 0.1, 0.3, 1.0]:
    p = OOF[wn] = oof(TR, wn); thr = pick_threshold(TR, p)
    cv.append(scorecard(TR, p, thr, f"W_NORMAL = {wn}", dense=("CinC normal", "healthy volunteers")))
print(f"{time.time() - t:.0f} s")
cv = pd.DataFrame(cv).set_index("model").T.round(3); cv

## 7.2 Final model and thresholds
Both thresholds come from the out-of-fold training predictions: the first catches ≥ 95 % of the real training events, the second ≥ 90 %.

In [ ]:
crit = [f"{s}: share of windows flagged" for s in ["CinC normal", "VTaC normal", "healthy volunteers"]]
W_NORMAL = float(cv.loc[crit].mean().idxmin().split("= ")[1])
p_oof = OOF[W_NORMAL]
THR = {"95% of events": pick_threshold(TR, p_oof, 0.95), "90% of events": pick_threshold(TR, p_oof, 0.90)}
model = fit_model(TR[FEATURES].to_numpy(float), TR.y.to_numpy(), weights(TR, W_NORMAL))
p_te = model.predict_proba(TE[FEATURES].to_numpy(float))[:, 1]
print("W_NORMAL:", W_NORMAL, "| thresholds:", THR, "| trained on", len(TR), "windows")

# 8. Test results (data never used for training or for any choice)
## 8.1 Window scores

In [ ]:
RES = pd.DataFrame([scorecard(TE, p_te, t, f"threshold {t} ({r})") for r, t in THR.items()]).set_index("model").T.round(3)
RES

In [ ]:
h = TE[TE.source == "healthy volunteers"].assign(p=p_te[(TE.source == "healthy volunteers").to_numpy()])
print("Healthy test volunteers, per activity (threshold", THR["95% of events"], "):")
for a, g in h.groupby("activity"):
    print(f"  {a:5s}: {len(g) * STRIDE / 3600:.1f} h | windows flagged {(g.p >= THR['95% of events']).mean():.1%} | highest score {g.p.max():.2f}")

## 8.2 From window scores to an emergency: three rules
This uses the test alarms with signal **after** the alarm: VTaC (up to 320 s) and the CinC test records that last 330 s. An event counts as detected if the rule fires on a window ending between 300 and 320 s.
* **1 window:** emergency as soon as one window is above the threshold.
* **2 in a row:** two consecutive windows above the threshold.
* **Hybrid (recommended):** one window ≥ 0.5, *or* two consecutive windows above the threshold. 0.5 is the natural "more likely than not" cut-off; it was not optimised on these test results.

In [ ]:
def detected_after(d, mdl, thr, rule, hi=HI, t_from=300, t_to=320):
    d = d.assign(p=mdl.predict_proba(d[FEATURES].to_numpy(float))[:, 1]).sort_values(["record", "t_end"])
    return pd.Series({rec: any(f and t >= t_from for t, f in rule_fires(zip(g.t_end, g.p), thr, rule, hi))
                      for rec, g in d[d.t_end <= t_to].groupby("record")})

lab_c = prim.set_index("record").assign(y=lambda x: ((x.label == 1) & x.alarm.isin(EVENTS)).astype(int)).y
lab_v = vt_meta.set_index("event").label
a_cinc330 = a_cinc[a_cinc.record.isin(a_cinc[a_cinc.t_end == 320].record)]

def rules_table(mdl, thr, p_test, name):
    rows = []
    for rule in ["1 window", "2 in a row", "hybrid"]:
        row = {"model": name, "rule": rule}
        for nm, d, lab in [("VTaC test", a_vtac, lab_v), ("CinC test 330 s", a_cinc330, lab_c)]:
            hit = detected_after(d, mdl, thr, rule); y = lab.reindex(hit.index)
            row[f"{nm}: real events detected"] = hit[y == 1].mean()
            row[f"{nm}: false alarms that trigger"] = hit[y == 0].mean()
        for s in ["CinC normal", "VTaC normal", "healthy volunteers"]:
            m = (TE.source == s).to_numpy()
            row[f"{s}: false triggers/h"] = rule_per_hour(TE[m], p_test[m], thr, rule)
        rows.append(row)
    return pd.DataFrame(rows).set_index(["model", "rule"]).T.round(3)

print("real events: VTaC test", int(lab_v.reindex(a_vtac.record.unique()).sum()), "| CinC 330-s test records", int(lab_c.reindex(a_cinc330.record.unique()).sum()))
RULES = rules_table(model, THR["95% of events"], p_te, "v3"); RULES

## 8.3 Per alarm type (test, single window at the alarm, the 95 % threshold)

In [ ]:
m = ((TE.kind == "alarm") & TE.primary).to_numpy()
d = TE[m].assign(p=p_te[m]); thr = THR["95% of events"]
PER = pd.DataFrame([{"source": s, "alarm type": a, "real emergencies": int(g.y.sum()), "caught": int((g.p[g.y == 1] >= thr).sum()),
                     "non-emergency alarms": int((g.y == 0).sum()), "ignored": int((g.p[g.y == 0] < thr).sum())}
                    for (s, a), g in d.groupby(["source", "alarm"])])
PER

## 8.4 Lead I (the steering-wheel lead)
The same test alarms scored with **lead I** instead of the usual lead: recorded lead I in VTaC (about a third of the alarms), lead I derived from two limb leads in CinC.

In [ ]:
rows = []
for name, d, lead in [("VTaC test, recorded lead I", vtac_alarm[vtac_alarm.split == "test"], "I"),
                      ("CinC test, derived lead I", cinc_alarm[cinc_alarm.record.isin(CINC_TEST)], "I*")]:
    recs = set(d[d.lead == lead].record)
    a = d[d.primary & d.record.isin(recs)].sort_values("record"); b = d[(d.lead == lead) & d.record.isin(recs)].sort_values("record")
    y = (a.label.to_numpy() == 1) & (a.alarm.isin(EVENTS).to_numpy() if "alarm" in a else True)
    pa, pb = model.predict_proba(a[FEATURES].to_numpy(float))[:, 1], model.predict_proba(b[FEATURES].to_numpy(float))[:, 1]
    for lab, pp in [(f"usual lead ({a.lead.mode()[0]})", pa), ("lead I", pb)]:
        rows.append({"test alarms": name, "n": len(a), "lead": lab, "AUC": roc_auc_score(y, pp) if 0 < y.sum() < len(y) else np.nan,
                     "real caught": (pp[y] >= THR["95% of events"]).mean() if y.any() else np.nan,
                     "same decision as usual lead": np.mean((pa >= THR["95% of events"]) == (pb >= THR["95% of events"]))})
pd.DataFrame(rows).round(3)

# 9. Optional: comparison with the v2 alarm checker
Runs only if the v2 model file exists (`V2_MODEL`, produced by the v2 notebook). v2 is scored with the new target, so it gets no credit for calling a real tachycardia an emergency.

In [ ]:
if os.path.exists(V2_MODEL):
    V2 = pickle.load(open(V2_MODEL, "rb")); thr2 = V2["thresholds"]["challenge"]
    p2_te = V2["model"].predict_proba(TE[V2["features"]].to_numpy(float))[:, 1]
    display(pd.DataFrame([scorecard(TE, p2_te, thr2, f"v2 (threshold {thr2})"), scorecard(TE, p_te, THR["95% of events"], f"v3 (threshold {THR['95% of events']})")])
            .set_index("model").T.round(3))
    norm = (TE.kind == "normal").to_numpy()
    def curve(p):
        out = []
        for t in np.round(np.arange(0.02, 0.9, 0.02), 2):
            sens = np.mean([(p[ev_mask(TE, s) & (TE.y == 1).to_numpy()] >= t).mean() for s in ["CinC", "VTaC"]])
            out.append((t, sens, rule_per_hour(TE[norm], p[norm], t)))
        return pd.DataFrame(out, columns=["thr", "sens", "per_h"])
    C2, C3 = curve(p2_te), curve(p_te)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(C2.sens, C2.per_h, "o-", label="v2 (alarm checker)"); ax.plot(C3.sens, C3.per_h, "o-", label="v3 (continuous monitor)")
    ax.set_xlim(0.7, 1.005); ax.set_yscale("log"); ax.grid(alpha=.3, which="both"); ax.legend()
    ax.set_xlabel("real events caught (test alarms)"); ax.set_ylabel("false triggers per hour (test normal signal)")
    ax.set_title("Trade-off at every threshold (lower = better)"); plt.tight_layout(); plt.savefig(f"{OUT}/tradeoff_v2_v3.png", dpi=130); plt.show()
    for s in [0.95, 0.90]:
        print(f"at >= {s:.0%} of test events caught: v2 {C2[C2.sens >= s].iloc[-1].per_h:.1f} -> v3 {C3[C3.sens >= s].iloc[-1].per_h:.1f} false triggers per hour")
    display(rules_table(V2["model"], thr2, p2_te, "v2"))
else:
    print("v2 model not found at", V2_MODEL, "- comparison skipped")

# 10. What the model relies on (SHAP)

In [ ]:
import shap
Xs = TR.sample(3000, random_state=0)[FEATURES].to_numpy(float)
svs = [shap.TreeExplainer(m).shap_values(Xs) for m in model.models]
sv = np.mean([s[1] if isinstance(s, list) else s for s in svs], axis=0)
imp = pd.Series(np.abs(sv).mean(0), index=FEATURES)
groups = {"ECG rhythm": ("ecg1", "ecg5", "ecgb", "ecgE", "ecg_hr"), "ECG quality / artefact": ("ecgq", "ecg_bsqi", "ecg_amp", "ecg_flat", "ecg_sat", "ecg_nan"),
          "VF detectors": ("vf8",), "QRS shape": ("qrs",), "PPG (pulse)": ("ppg",), "ECG-PPG cross-check": ("x_",)}
share = pd.Series({g: imp[[f for f in FEATURES if f.startswith(p)]].sum() for g, p in groups.items()})
print((share / share.sum() * 100).round(1).sort_values(ascending=False).to_string())
print("\ntop 10 features:", ", ".join(imp.sort_values(ascending=False).index[:10]))

# 11. Save the model and use it in the car
`Monitor.step()` is called every 10 s with the last 60 s of signal (`sig`: array of shape (15000, channels) at 250 Hz, `names`: channel names, `lead`: the ECG channel). It averages the 5 models and applies the hybrid rule. The saved file contains only standard LightGBM models (`bundle["models"]`), so it loads anywhere LightGBM is installed.

In [ ]:
bundle = {"models": model.models, "features": FEATURES, "thresholds": THR, "rule": "hybrid", "high_threshold": HI, "stride_s": STRIDE, "window_s": 60, "fs": FS,
          "target": "asystole, extreme bradycardia, VT, VF (real tachycardia = not an emergency)", "w_normal": W_NORMAL,
          "trained_on": "CinC 2015 + VTaC alarms, their pre-alarm minutes (false-alarm records), healthy volunteers (PTT)",
          "versions": {"lightgbm": lgb.__version__, "sklearn": sklearn.__version__, "wfdb": wfdb.__version__}}
pickle.dump(bundle, open(f"{OUT}/cabinguard_monitor_v3.pkl", "wb"))

class Monitor:
    """Emergency = one window >= high_threshold, or 2 consecutive windows >= threshold."""
    def __init__(self, bundle, operating_point="95% of events"):
        self.b, self.thr, self.hi, self.run = bundle, bundle["thresholds"][operating_point], bundle["high_threshold"], 0
    def step(self, sig60, names, lead):
        f = record_features(sig60, names, lead)
        x = np.array([[f.get(c, np.nan) for c in self.b["features"]]], float)
        p = float(np.mean([m.predict_proba(x)[0, 1] for m in self.b["models"]]))
        self.run = self.run + 1 if p >= self.thr else 0
        return {"p_emergency": round(p, 3), "emergency": p >= self.hi or self.run >= 2}

def stream(sig, names, lead, t_from, t_to, label):
    mon = Monitor(bundle); out = []
    for e in range(t_from, t_to + 1, STRIDE):
        r = mon.step(sig[(e - 60) * FS:e * FS], names, lead); out.append(f"{e}s:{r['p_emergency']:.2f}{'!' if r['emergency'] else ''}")
    print(f"{label}\n   " + "  ".join(out))

t0, n_steps = time.perf_counter(), 0
ensure_ptt("s3_run"); sig, names = load_ptt("s3_run")
stream(sig, names, "ECG", 60, 300, "healthy test volunteer s3, running, 60-300 s:"); n_steps += 25
for lab in [1, 0]:
    rec = prim[prim.record.isin(CINC_TEST) & (prim.alarm == "Ventricular_Tachycardia") & (prim.label == lab) & prim.record.str.endswith("l")].iloc[0]
    for e in ("hea", "mat"):
        download(f"{CINC_URL}/{rec.record}.{e}", f"{CINC_DIR}/{rec.record}.{e}")
    r = wfdb.rdrecord(f"{CINC_DIR}/{rec.record}")
    stream(r.p_signal, list(r.sig_name), rec.lead, 250, 330, f"CinC test {rec.record}: {'REAL' if lab else 'FALSE'} VT alarm at 300 s, 250-330 s:"); n_steps += 9
print("('!' = emergency raised)")
print(f"\n{(time.perf_counter() - t0) / n_steps * 1000:.0f} ms per 10-s step on {platform.machine()} | model file "
      f"{os.path.getsize(f'{OUT}/cabinguard_monitor_v3.pkl') / 1024:.0f} KB -> {OUT}/cabinguard_monitor_v3.pkl")

# 12. Limitations
* **"Normal" is not certified normal.** The ICU minutes have no rhythm annotation. Some contain real arrhythmias that are not emergencies (ectopic beats, atrial fibrillation), so the ICU false-trigger rates are a worst case.
* **VTaC patients are VT-prone.** The database drew 5,037 alarms from 18,465 VT alarms in 2,376 recordings, so the minutes before a false VT alarm often contain ventricular ectopy.
* **Healthy volunteers are not drivers.** They wore a finger clip while sitting, walking and running, not a steering-wheel sensor or a camera. The healthy test set is about 2.5 h: zero triggers there means the true rate is probably below about 1.2 per hour (rule of three), not that it is zero.
* **Persistence.** Strict "2 in a row" misses about a quarter of the real VT alarms, because many are short runs that stop within 10 s. The hybrid rule keeps strong single detections, and waits at most 10 s for weaker ones.
* **VF is rare:** 1 real VF in the CinC test set.
* **Not yet validated in a vehicle.** In-car recordings with the actual sensors are the next step.